In [8]:
import vectorbt as vbt
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def load_data(path):
    return pd.read_csv(path, index_col="timestamp", parse_dates=True)

In [9]:
bh_path = "../models/gru_tick_2160/signal/bh_signal.csv"
best_model_path = "../models/gru_tick_2160/signal/best_signal.csv"
normal_model_path = "../models/gru_tick_2160/signal/normal_signal.csv"
bh_data = load_data(path=bh_path)
best_model_data = load_data(path=best_model_path)
normal_model_data = load_data(path=normal_model_path)

In [10]:
def get_return(df: pd.DataFrame):
    data = df["equity"].resample("1h").ffill()
    data = data.pct_change().fillna(0)
    return data
bh_returns = get_return(bh_data)
b_returns = get_return(best_model_data)
n_returns = get_return(normal_model_data)

In [11]:

def evaluate_model_vs_benchmark(model_returns: pd.Series, benchmark_returns: pd.Series, 
                                model_name: str = "Agent", benchmark_name: str= "Benchmark"):
    # 1. Đồng bộ và xử lý dữ liệu
    model_returns = model_returns.asfreq('1h').fillna(0) if model_returns.index.freq is None else model_returns
    benchmark_returns = benchmark_returns.asfreq('1h').fillna(0) if benchmark_returns.index.freq is None else benchmark_returns

    # 2. Tính toán Stats
    annual_rf = 0.052
    hourly_rf = (1 + annual_rf)**(1/8760) - 1
    stats = model_returns.vbt.returns(freq='1h').stats(
        settings=dict(benchmark_rets=benchmark_returns, risk_free=hourly_rf, freq="1h")
    )

    # 3. Tính Lợi nhuận tích lũy (Chuyển sang phần trăm %)
    df_plot = pd.DataFrame({model_name: model_returns, benchmark_name: benchmark_returns})
    # Công thức: (Tích lũy lợi nhuận - 1) * 100 để ra %
    cum_returns_pct = ((1 + df_plot).cumprod() - 1) * 100

    # 4. Tạo Figure (1 hàng, 1 cột)
    fig = vbt.make_subplots(rows=1, cols=1)

    # --- Vẽ đường Agent và Benchmark ---
    cum_returns_pct[model_name].vbt.plot(fig=fig, 
                                         trace_kwargs=dict(name=model_name, line=dict(color='#1f77b4', width=2)))
    cum_returns_pct[benchmark_name].vbt.plot(fig=fig, 
                                             trace_kwargs=dict(name=benchmark_name, line=dict(color='#ff7f0e', width=2)))

    # 5. Tinh chỉnh Layout gọn gàng
    fig.update_layout(
        title_text="So sánh Lợi nhuận tích lũy (%)",
        title_font=dict(size=18),
        height=400,  
        width=900,   # Thu hẹp width lại một chút vì chỉ còn 1 biểu đồ
        template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5), # Căn giữa legend
        margin=dict(l=60, r=40, t=80, b=60),
        hovermode="x unified" # Hiển thị tooltip so sánh cả 2 đường cùng lúc khi rê chuột
    )

    # Cập nhật nhãn trục
    fig.update_yaxes(title_text="Lợi nhuận (%)")
    fig.update_xaxes(title_text="Thời gian")

    return stats, fig

normal_stats, fig = evaluate_model_vs_benchmark(
    model_returns=n_returns, 
    benchmark_returns=bh_returns, 
)
fig.show()

In [12]:
normal_stats

Start                        2024-09-02 00:00:00
End                          2026-03-31 00:00:00
Period                         575 days 01:00:00
Total Return [%]                       17.512422
Benchmark Return [%]                   16.951414
Annualized Return [%]                  10.785964
Annualized Volatility [%]              10.816302
Max Drawdown [%]                       15.062878
Max Drawdown Duration          182 days 18:00:00
Sharpe Ratio                            0.532379
Calmar Ratio                            0.716063
Omega Ratio                             1.104298
Sortino Ratio                             1.4875
Skew                                    0.821111
Kurtosis                              263.028434
Tail Ratio                                   inf
Common Sense Ratio                           inf
Value at Risk                                0.0
Alpha                                   0.049735
Beta                                    0.057562
Name: equity, dtype:

In [13]:
# 2. Test với Best Model
best_stats, fig = evaluate_model_vs_benchmark(
    model_returns=b_returns, 
    benchmark_returns=bh_returns, 
)
fig.show()

In [14]:
def evaluate_strategy_returns(strategy_returns: pd.Series, strategy_name: str = "Strategy"):
    """
    Đánh giá và vẽ biểu đồ lợi nhuận cho một chiến lược đơn (không có benchmark).
    """
    # 1. Đồng bộ và xử lý dữ liệu
    strategy_returns = strategy_returns.asfreq('1h').fillna(0) if strategy_returns.index.freq is None else strategy_returns

    # 2. Tính toán Stats (Chỉ dùng lợi nhuận của chiến lược)
    annual_rf = 0.052
    hourly_rf = (1 + annual_rf)**(1/8760) - 1
    
    # Đã bỏ tham số benchmark_rets ra khỏi phần settings
    stats = strategy_returns.vbt.returns(freq='1h').stats(
        settings=dict(risk_free=hourly_rf, freq="1h")
    )

    # 3. Tính Lợi nhuận tích lũy (Chuyển sang phần trăm %)
    # Công thức: (Tích lũy lợi nhuận - 1) * 100 để ra %
    cum_returns_pct = ((1 + strategy_returns).cumprod() - 1) * 100

    # 4. Tạo Figure (1 hàng, 1 cột)
    fig = vbt.make_subplots(rows=1, cols=1)

    # --- Vẽ đường Lợi nhuận của chiến lược ---
    cum_returns_pct.vbt.plot(fig=fig, 
                             trace_kwargs=dict(name=strategy_name, line=dict(color='#1f77b4', width=2)))

    # 5. Tinh chỉnh Layout gọn gàng
    fig.update_layout(
        title_text=f"Lợi nhuận tích lũy (%) - {strategy_name}",
        title_font=dict(size=18),
        height=400,  
        width=900,
        template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5), # Căn giữa legend
        margin=dict(l=60, r=40, t=80, b=60),
        hovermode="x unified" # Hiển thị tooltip chi tiết khi rê chuột
    )

    # Cập nhật nhãn trục
    fig.update_yaxes(title_text="Lợi nhuận (%)")
    fig.update_xaxes(title_text="Thời gian")

    return stats, fig

In [15]:
stats, fig = evaluate_strategy_returns(strategy_returns=bh_returns, strategy_name="B&H")
stats

d:\Study\MyPaper\drl_crypto\venv\Lib\site-packages\vectorbt\generic\stats_builder.py:396: UserWarning: Metric 'benchmark_return' requires benchmark_rets to be set
  warnings.warn(warning_message)
d:\Study\MyPaper\drl_crypto\venv\Lib\site-packages\vectorbt\generic\stats_builder.py:396: UserWarning: Metric 'alpha' requires benchmark_rets to be set
  warnings.warn(warning_message)
d:\Study\MyPaper\drl_crypto\venv\Lib\site-packages\vectorbt\generic\stats_builder.py:396: UserWarning: Metric 'beta' requires benchmark_rets to be set
  warnings.warn(warning_message)


Start                        2024-09-02 00:00:00
End                          2026-03-31 00:00:00
Period                         575 days 01:00:00
Total Return [%]                       16.951414
Annualized Return [%]                  10.449961
Annualized Volatility [%]              46.560549
Max Drawdown [%]                       50.083794
Max Drawdown Duration          175 days 06:00:00
Sharpe Ratio                            0.337511
Calmar Ratio                             0.20865
Omega Ratio                             1.011037
Sortino Ratio                             0.6249
Skew                                   -0.132938
Kurtosis                                9.303273
Tail Ratio                              0.992123
Common Sense Ratio                      1.095799
Value at Risk                          -0.007363
Name: equity, dtype: object